# Runtime model selection and out-of-the-box telemetry for a hosted agent

This notebook answers two questions I've been asked about Foundry hosted agents, using the
GitHub Copilot SDK agent from [`08-10`](../08-10-hosted-copilot-sdk-agent/08-10-01-deploy-hosted-copilot-sdk-agent.ipynb)
as the basis:

1. **Can a chat UI change the agent's model or reasoning effort mid-conversation, at runtime?**
   The user picks a smaller, faster model or a smarter one in the UI, and later changes the
   reasoning effort. The change must apply to the running agent without creating a new agent
   version or redeploying.
2. **What logging and telemetry does a hosted agent produce out of the box?**

**Short answers.**

- **Runtime switching: yes, in the container code.** A hosted agent version is immutable,
  environment variables included, so the platform itself has no "change the model" switch. But the
  invocations protocol passes the request body to your code unchanged, and the Copilot SDK can
  switch the model and reasoning effort of a live session with `session.set_model()`, keeping the
  conversation history. The container reads `model` and `reasoning_effort` from each request and
  applies them. The agent version, image and deployment stay the same.
- **Telemetry: three layers.** The Foundry platform records every invocation in Application
  Insights without any code. The container's stdout and stderr are available through a
  per-session log stream. Per-model-call detail (model, reasoning effort, tokens) needs a small
  amount of tracing code, which this agent includes.

This lab is **self-contained**: it provisions its own Foundry account, project, container
registry and Application Insights, and ships its own copy of the container source. It borrows the
agent design from 08-10 but depends on nothing that 08-10 deployed.

By the end you will have:

- A Foundry stack in its own resource group, with three model deployments for the UI to choose
  from: `gpt-5.4-nano` (faster), `gpt-5.4-mini` (default), `gpt-5.4` (smarter).
- A hosted agent `github-copilot-runtime-model` whose container applies per-request model and
  reasoning effort selections.
- A simulated chat where the "UI" switches model and reasoning effort between turns, with proof
  from the agent's own events that each turn ran on the selected model and effort.
- Proof that the agent still has one version and one image after all the switches.
- A walk through the out-of-the-box telemetry: injected environment variables, the session log
  stream, and what lands in Application Insights.

## Prerequisites

1. **`az` CLI** logged in with `az login`, with rights to create a resource group, a Foundry
   account and role assignments in the subscription.
2. **Quota** in `swedencentral` for `gpt-5.4-nano`, `gpt-5.4-mini` and `gpt-5.4` GlobalStandard
   (this notebook deploys 50K TPM each).
3. **Python packages** from the repo's `uv` environment: `azure-ai-projects>=2.1.0`,
   `azure-identity`, `requests`.

No other notebook has to run first.

Run this notebook from its own folder (`08-agents/08-10c-hosted-copilot-sdk-agent-runtime-model/`)
so the relative path to `src/` resolves.

In [2]:
import json
import shutil
import subprocess

def sh(cmd, **kw):
    # Run a shell command, stream output, return CompletedProcess.
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=False, **kw)

if shutil.which("az") is None:
    raise AssertionError("`az` is not on PATH. Install the Azure CLI and restart the kernel.")

who = subprocess.run(
    "az account show --query '{name:name, id:id, user:user.name}' -o jsonc",
    shell=True, capture_output=True, text=True,
)
if who.returncode != 0:
    raise AssertionError(f"Not logged in to Azure CLI. Run `az login`.\n{who.stderr.strip()}")
print(who.stdout)

{
  "id": "00000000-0000-0000-0000-000000000000",
  "name": "{subscription-name}",
  "user": "{user}@{tenant}.onmicrosoft.com"
}



## Step 1 - Variables and principal id

Names follow this repo's convention: a 6-char suffix derived from the subscription and environment
name keeps the globally-unique account FQDN distinct across developers.

- `DEFAULT_MODEL` and `DEFAULT_REASONING_EFFORT` are what a new conversation starts with.
- `MODEL_CHOICES` is what the chat UI offers. The same list is passed to the container as
  `AZURE_AI_ALLOWED_MODELS`, so a request for any other deployment is rejected.

In [3]:
import base64
import hashlib
import pathlib
import re

ENV_NAME = "foundry-copilot-sdk-08-10c"
LOCATION = "swedencentral"
RESOURCE_GROUP = f"rg-{ENV_NAME}"
DEPLOY_NAME = f"deploy-{ENV_NAME}"
AGENT_NAME = "github-copilot-runtime-model"

# What the chat UI offers. Versions are the GA versions in swedencentral.
MODEL_CHOICES = {
    "gpt-5.4-nano": "2026-03-17",   # smaller, faster
    "gpt-5.4-mini": "2026-03-17",   # default
    "gpt-5.4":      "2026-03-05",   # smarter
}
DEFAULT_MODEL = "gpt-5.4-mini"
DEFAULT_REASONING_EFFORT = "medium"
MODEL_CAPACITY = 50                 # thousand tokens per minute per deployment

SUBSCRIPTION_ID = subprocess.run("az account show --query id -o tsv", shell=True,
                                 capture_output=True, text=True).stdout.strip()
SUFFIX = hashlib.sha256((SUBSCRIPTION_ID + ENV_NAME).encode()).hexdigest()[:6]
AI_ACCOUNT_NAME = f"aif-copilot-rt-{SUFFIX}"
AI_PROJECT_NAME = f"project-copilot-rt-{SUFFIX}"

# Principal id from the cached JWT, which avoids a Graph round-trip
token = subprocess.run("az account get-access-token --query accessToken -o tsv", shell=True,
                       capture_output=True, text=True).stdout.strip()
PRINCIPAL_ID = json.loads(base64.urlsafe_b64decode(token.split(".")[1] + "=="))["oid"]

print(f"Resource group: {RESOURCE_GROUP}")
print(f"Location:       {LOCATION}")
print(f"AI account:     {AI_ACCOUNT_NAME}")
print(f"AI project:     {AI_PROJECT_NAME}")
print(f"Agent name:     {AGENT_NAME}")
print(f"Model choices:  {list(MODEL_CHOICES)} (default {DEFAULT_MODEL}, effort {DEFAULT_REASONING_EFFORT})")
print(f"Principal id:   {PRINCIPAL_ID}")

Resource group: rg-foundry-copilot-sdk-08-10c
Location:       swedencentral
AI account:     aif-copilot-rt-79a3fa
AI project:     project-copilot-rt-79a3fa
Agent name:     github-copilot-runtime-model
Model choices:  ['gpt-5.4-nano', 'gpt-5.4-mini', 'gpt-5.4'] (default gpt-5.4-mini, effort medium)
Principal id:   00000000-0000-0000-0000-000000000000


## Step 2 - Provision the Foundry stack with Bicep

`infra/main.bicep` is the same subscription-scoped template 08-10 uses, so this lab stands on its
own. It creates the resource group, an AI Services account and Foundry project, the **capability
host** that runs hosted agents, a container registry connected to the project, and Application
Insights plus Log Analytics. Application Insights matters here: connecting it to the project is
what makes the platform inject `APPLICATIONINSIGHTS_CONNECTION_STRING` into the container, which
the telemetry in Step 8 depends on.

All three model choices are deployed here in one pass, so the UI can switch between them.

Provisioning takes **5-10 minutes**. Re-running is safe; it is an incremental deployment.

In [4]:
bicep_params = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "environmentName":          {"value": ENV_NAME},
        "location":                 {"value": LOCATION},
        "aiDeploymentsLocation":    {"value": LOCATION},
        "aiFoundryResourceName":    {"value": AI_ACCOUNT_NAME},
        "aiFoundryProjectName":     {"value": AI_PROJECT_NAME},
        "principalId":              {"value": PRINCIPAL_ID},
        "principalType":            {"value": "User"},
        "aiProjectDeploymentsJson": {"value": json.dumps([
            {
                "name": name,
                "model": {"format": "OpenAI", "name": name, "version": version},
                "sku": {"name": "GlobalStandard", "capacity": MODEL_CAPACITY},
            }
            for name, version in MODEL_CHOICES.items()
        ])},
        "enableHostedAgents":       {"value": True},
        "enableCapabilityHost":     {"value": True},
        "enableMonitoring":         {"value": True},
    },
}
PARAMS_FILE = pathlib.Path("infra") / ".main.parameters.runtime.json"
PARAMS_FILE.write_text(json.dumps(bicep_params, indent=2))

r = sh(
    f'az deployment sub create --name "{DEPLOY_NAME}" --location "{LOCATION}" '
    f'--template-file infra/main.bicep --parameters @"{PARAMS_FILE}" -o none'
)
assert r.returncode == 0, "bicep deployment failed - see the output above"

outputs = {k.upper(): v["value"] for k, v in json.loads(subprocess.run(
    f'az deployment sub show --name "{DEPLOY_NAME}" --query properties.outputs -o json',
    shell=True, capture_output=True, text=True,
).stdout).items()}

PROJECT_ENDPOINT = outputs["AZURE_AI_PROJECT_ENDPOINT"]
ACCOUNT_NAME     = outputs["AZURE_AI_ACCOUNT_NAME"]
PROJECT_NAME     = outputs["AZURE_AI_PROJECT_NAME"]
ACR_LOGIN_SERVER = outputs["AZURE_CONTAINER_REGISTRY_ENDPOINT"]
ACR_NAME         = ACR_LOGIN_SERVER.split(".")[0]
APPINSIGHTS_APP_ID = re.search(r"ApplicationId=([0-9a-f-]+)",
                               outputs["APPLICATIONINSIGHTS_CONNECTION_STRING"]).group(1)

print(f"Project endpoint: {PROJECT_ENDPOINT}")
print(f"ACR:              {ACR_LOGIN_SERVER}")
print("Model deployments:", subprocess.run(
    f"az cognitiveservices account deployment list -g {RESOURCE_GROUP} -n {ACCOUNT_NAME} "
    f"--query '[].name' -o tsv", shell=True, capture_output=True, text=True).stdout.split())

$ az deployment sub create --name "deploy-foundry-copilot-sdk-08-10c" --location "swedencentral" --template-file infra/main.bicep --parameters @"infra/.main.parameters.runtime.json" -o none


/home/jp/development/corticalstack/awesome-foundry-nextgen/08-agents/08-10c-hosted-copilot-sdk-agent-runtime-model/infra/core/ai/ai-project.bicep(181,58) : Warning BCP318: The value of type "module | null" may be null at the start of the deployment, which would cause this access expression (and the overall deployment with it) to fail. [https://aka.ms/bicep/core-diagnostics#BCP318]
/home/jp/development/corticalstack/awesome-foundry-nextgen/08-agents/08-10c-hosted-copilot-sdk-agent-runtime-model/infra/core/ai/ai-project.bicep(185,57) : Warning BCP318: The value of type "module | null" may be null at the start of the deployment, which would cause this access expression (and the overall deployment with it) to fail. [https://aka.ms/bicep/core-diagnostics#BCP318]
/home/jp/development/corticalstack/awesome-foundry-nextgen/08-agents/08-10c-hosted-copilot-sdk-agent-runtime-model/infra/core/ai/ai-project.bicep(189,64) : Warning BCP318: The value of type "module | null" may be null at the start o

Project endpoint: https://aif-copilot-rt-79a3fa.services.ai.azure.com/api/projects/project-copilot-rt-79a3fa
ACR:              crkvlyutopnoraa.azurecr.io
Model deployments: ['gpt-5.4-nano', 'gpt-5.4-mini', 'gpt-5.4']


## Step 3 - Build the agent image

The container in [`src/github-copilot-invocations/`](src/github-copilot-invocations/) is this lab's
own copy of the 08-10 agent, with one addition in `main.py`. Each request may carry `model` and `reasoning_effort` next to
`input`:

```json
{"input": "What is my name?", "model": "gpt-5.4-nano", "reasoning_effort": "low"}
```

When the requested values differ from the live Copilot session, the handler calls
`session.set_model(model, reasoning_effort=...)` before sending the prompt. The Copilot SDK applies
the change from the next model call and keeps the conversation history. Unknown models or effort
values get HTTP 400. `tracing.py` also records the reasoning effort on each `chat <model>` span.

In [5]:
r = sh(
    f'az acr build --registry "{ACR_NAME}" '
    f'--image "{AGENT_NAME}:latest" '
    f'--platform linux/amd64 '
    f'./src/github-copilot-invocations/ -o none'
)
assert r.returncode == 0, "az acr build failed - see the output above"

$ az acr build --registry "crkvlyutopnoraa" --image "github-copilot-runtime-model:latest" --platform linux/amd64 ./src/github-copilot-invocations/ -o none


2026/09/22 09:22:08 Downloading source code...
2026/09/22 09:22:09 Finished downloading source code
2026/09/22 09:22:09 Using acb_vol_00000000-0000-0000-0000-000000000000 as the home volume
2026/09/22 09:22:09 Setting up Docker configuration...
2026/09/22 09:22:10 Successfully set up Docker configuration
2026/09/22 09:22:10 Logging in to registry: crkvlyutopnoraa.azurecr.io
2026/09/22 09:22:11 Successfully logged into crkvlyutopnoraa.azurecr.io
2026/09/22 09:22:11 Executing step ID: build. Timeout(sec): 28800, Working directory: '', Network: ''
2026/09/22 09:22:11 Scanning for dependencies...
2026/09/22 09:22:11 Successfully scanned dependencies
2026/09/22 09:22:11 Launching container with name: build
Sending build context to Docker daemon  32.26kB
Step 1/7 : FROM mcr.microsoft.com/devcontainers/python:3.12-bookworm
3.12-bookworm: Pulling from devcontainers/python
abf56b2f8724: Pulling fs layer
08457856946d: Pulling fs layer
8cab6ce149c2: Pulling fs layer
01a6a9ffe665: Pulling fs layer

## Step 4 - Register the hosted agent

Same two-pass registration as 08-10. The first version exposes the per-agent managed identity's
`client_id`; the second version pins it as `AZURE_CLIENT_ID`, and the bootstrap version is deleted.
If you re-run this notebook, older versions of this agent are deleted too (with `force=true`, which
also deletes their sessions), so the agent is left with exactly one version.

The environment variables carry the **defaults** only. They are fixed for the life of the version,
which is why runtime changes have to travel in the request instead:

| Variable | Value |
|---|---|
| `AZURE_AI_MODEL_DEPLOYMENT_NAME` | default model for a new conversation |
| `AZURE_AI_REASONING_EFFORT` | default reasoning effort |
| `AZURE_AI_ALLOWED_MODELS` | deployments a request may switch to |

In [6]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import AgentProtocol, HostedAgentDefinition, ProtocolVersionRecord
from azure.core.rest import HttpRequest
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True)
CONTAINER_IMAGE = f"{ACR_LOGIN_SERVER}/{AGENT_NAME}:latest"

def agent_definition(extra_env=None):
    env = {
        "AZURE_AI_PROJECT_ENDPOINT":      PROJECT_ENDPOINT,
        "AZURE_AI_MODEL_DEPLOYMENT_NAME": DEFAULT_MODEL,
        "AZURE_AI_REASONING_EFFORT":      DEFAULT_REASONING_EFFORT,
        "AZURE_AI_ALLOWED_MODELS":        ",".join(MODEL_CHOICES),
        **(extra_env or {}),
    }
    return HostedAgentDefinition(
        container_protocol_versions=[ProtocolVersionRecord(protocol=AgentProtocol.INVOCATIONS, version="1.0.0")],
        cpu="2",
        memory="4Gi",
        image=CONTAINER_IMAGE,
        environment_variables=env,
    )

def version_metadata(version):
    return json.loads(client.send_request(
        HttpRequest("GET", f"/agents/{AGENT_NAME}/versions/{version}?api-version=v1")
    ).text())

bootstrap = client.agents.create_version(agent_name=AGENT_NAME, definition=agent_definition())
AGENT_IDENTITY_CLIENT_ID = version_metadata(bootstrap.version)["instance_identity"]["client_id"]

agent = client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=agent_definition({"AZURE_CLIENT_ID": AGENT_IDENTITY_CLIENT_ID}),
)
client.agents.delete_version(AGENT_NAME, agent_version=bootstrap.version)
AGENT_VERSION = agent.version
print(f"Active version: {AGENT_NAME} v{AGENT_VERSION} (bootstrap v{bootstrap.version} deleted)")

# Re-runs: remove versions from earlier runs, including any sessions still attached to them.
earlier = json.loads(client.send_request(
    HttpRequest("GET", f"/agents/{AGENT_NAME}/versions?api-version=v1")).text()).get("data", [])
for v in earlier:
    if v["version"] != AGENT_VERSION:
        client.send_request(HttpRequest(
            "DELETE", f"/agents/{AGENT_NAME}/versions/{v['version']}?api-version=v1&force=true"))
        print(f"Deleted earlier version v{v['version']}")
print(f"Image:          {CONTAINER_IMAGE}")

Active version: github-copilot-runtime-model v4 (bootstrap v3 deleted)
Deleted earlier version v2
Image:          crkvlyutopnoraa.azurecr.io/github-copilot-runtime-model:latest


## Step 5 - Grant runtime roles to the agent identity

The container calls every model through `<project>/openai/v1/responses` with its managed identity,
so one set of account-level roles covers all three model choices.

In [7]:
import time

AGENT_IDENTITY_PRINCIPAL_ID = version_metadata(AGENT_VERSION)["instance_identity"]["principal_id"]
acr_id = subprocess.check_output(f"az acr show -n {ACR_NAME} --query id -o tsv", shell=True, text=True).strip()
account_id = (f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
              f"/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}")
project_id = f"{account_id}/projects/{PROJECT_NAME}"

grants = [
    ("AcrPull",                              acr_id),      # pull the image
    ("53ca6127-db72-4b80-b1b0-d745d6d5456d", project_id),  # Foundry User
    ("5e0bd9bd-7b93-4f28-af87-19fc36ad61bd", account_id),  # Cognitive Services OpenAI User
    ("a97b65f3-24c7-4388-baec-2e87135dc908", account_id),  # Cognitive Services User
]
for role, scope in grants:
    r = subprocess.run(
        f"az role assignment create --assignee-object-id {AGENT_IDENTITY_PRINCIPAL_ID} "
        f"--assignee-principal-type ServicePrincipal --role '{role}' --scope '{scope}'",
        shell=True, capture_output=True, text=True,
    )
    status = "granted" if r.returncode == 0 else ("already granted" if "already exist" in r.stderr.lower() else "FAILED")
    print(f"  {status}: {role}")

print("Waiting 90s for RBAC propagation and container cold start...")
time.sleep(90)

  granted: AcrPull
  granted: 53ca6127-db72-4b80-b1b0-d745d6d5456d
  granted: 5e0bd9bd-7b93-4f28-af87-19fc36ad61bd
  granted: a97b65f3-24c7-4388-baec-2e87135dc908
Waiting 90s for RBAC propagation and container cold start...


## Step 6 - A chat client that behaves like the UI

`chat()` plays the part of the chat UI's backend:

- `ui` holds the user's current selection in the UI. **Every request sends it**, not only when it
  changes. Hosted agent sessions are torn down after an idle timeout (15 minutes by default) and
  restored on the next request; the conversation survives because it lives in `$HOME`, but the
  container's in-memory selection resets to the defaults. Sending the selection each time keeps
  the UI and the agent in step.
- The first request has no session id; the platform creates a session and returns its id, which
  every later turn passes as `agent_session_id`.
- From the streamed events it collects `session.model_change` (a switch happened) and
  `assistant.usage` (which model and reasoning effort each model call actually used, with tokens
  and latency).

In [8]:
import requests

ui = {"model": DEFAULT_MODEL, "reasoning_effort": DEFAULT_REASONING_EFFORT}
SESSION_ID = None
TURNS = []

def _headers(**extra):
    token = credential.get_token("https://ai.azure.com/.default").token
    return {"Authorization": f"Bearer {token}", "Foundry-Features": "HostedAgents=V1Preview", **extra}

def chat(message, retries=5, retry_delay=30):
    # Send one chat turn with the UI's current model and reasoning effort selection.
    global SESSION_ID
    url = f"{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/endpoint/protocols/invocations?api-version=v1"
    if SESSION_ID:
        url += f"&agent_session_id={SESSION_ID}"
    body = {"input": message, **ui}
    for attempt in range(1, retries + 1):
        started = time.time()
        resp = requests.post(url, headers=_headers(**{"Content-Type": "application/json"}),
                             json=body, stream=True, timeout=600)
        if resp.status_code >= 500 and attempt < retries:
            print(f"  [HTTP {resp.status_code}, attempt {attempt}/{retries}; retrying in {retry_delay}s]")
            time.sleep(retry_delay)
            continue
        if resp.status_code != 200:
            return {"status": resp.status_code, "error": resp.json()}
        text, changes, usage, error = [], [], [], None
        for line in resp.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:"):
                continue
            event = json.loads(line[5:].strip())
            kind, data = event.get("type"), event.get("data") or {}
            if kind == "assistant.message_delta":
                text.append(data.get("deltaContent", ""))
            elif kind == "session.model_change":
                changes.append(data)
            elif kind == "assistant.usage":
                usage.append(data)
            elif kind == "error":
                error = event.get("message")
            elif "session_id" in event and "invocation_id" in event:
                SESSION_ID = event["session_id"]
        if error and attempt < retries:
            print(f"  [stream error, attempt {attempt}/{retries}; retrying in {retry_delay}s]")
            time.sleep(retry_delay)
            continue
        turn = {"ui": dict(ui), "message": message, "reply": "".join(text).strip(),
                "changes": changes, "usage": usage, "seconds": round(time.time() - started, 1)}
        TURNS.append(turn)
        return turn
    raise RuntimeError("chat turn failed after retries")

def show(turn):
    print(f"UI selection : model={turn['ui']['model']}  reasoning_effort={turn['ui']['reasoning_effort']}")
    for c in turn["changes"]:
        print(f"model_change : {c.get('previousModel')} ({c.get('previousReasoningEffort')}) "
              f"-> {c.get('newModel')} ({c.get('reasoningEffort')})")
    for u in turn["usage"]:
        print(f"model call   : model={u.get('model')}  reasoning_effort={u.get('reasoningEffort')}  "
              f"reasoning_tokens={u.get('reasoningTokens')}  output_tokens={u.get('outputTokens')}  "
              f"ttft_ms={u.get('ttftMs')}")
    print(f"reply ({turn['seconds']}s): {turn['reply']}")

def agent_versions():
    listing = json.loads(client.send_request(
        HttpRequest("GET", f"/agents/{AGENT_NAME}/versions?api-version=v1")).text())
    return [(v["version"], v.get("status"), v["definition"]["image"].split("/")[-1],
             v["definition"]["environment_variables"]["AZURE_AI_MODEL_DEPLOYMENT_NAME"])
            for v in listing.get("data", [])]

print("Helpers defined: chat(), show(), agent_versions()")

Helpers defined: chat(), show(), agent_versions()


## Step 7 - The conversation

### 7.1 Start on the defaults

Record the agent's versions before any switch, then open the conversation on the default model and
reasoning effort.

In [9]:
from datetime import datetime, timezone

CONVERSATION_START = datetime.now(timezone.utc).isoformat()
VERSIONS_BEFORE = agent_versions()
print("Agent versions before the conversation:", VERSIONS_BEFORE)
print()
show(chat("Hi, I'm Priya. I'm planning a team offsite for 12 people in Lisbon in October. "
          "Remember those details. Reply with one short sentence."))

Agent versions before the conversation: [('4', 'active', 'github-copilot-runtime-model@sha256:afbac516f0d670b8990ce1f9c792e562c5cb680b409de776ca474d061d6e5470', 'gpt-5.4-mini')]

UI selection : model=gpt-5.4-mini  reasoning_effort=medium
model call   : model=gpt-5.4-mini  reasoning_effort=medium  reasoning_tokens=112.0  output_tokens=234.0  ttft_ms=3455.0
model call   : model=gpt-5.4-mini  reasoning_effort=medium  reasoning_tokens=0.0  output_tokens=20.0  ttft_ms=1036.0
reply (11.8s): Got it—I’ll remember Priya, 12 people, Lisbon, October.


### 7.2 The user picks a smaller, faster model

In the UI the user switches the model to `gpt-5.4-nano`. The reasoning effort selection stays at
`medium`. The next request carries the new selection, the container switches the live session, and
the answer shows the conversation history carried over to the new model.

In [10]:
ui["model"] = "gpt-5.4-nano"
show(chat("How many people are coming, and which city? One line."))

UI selection : model=gpt-5.4-nano  reasoning_effort=medium
model_change : gpt-5.4-mini (medium) -> gpt-5.4-nano (medium)
model call   : model=gpt-5.4-nano  reasoning_effort=medium  reasoning_tokens=0.0  output_tokens=126.0  ttft_ms=2129.0
model call   : model=gpt-5.4-nano  reasoning_effort=medium  reasoning_tokens=0.0  output_tokens=15.0  ttft_ms=1249.0
reply (4.5s): 12 people are coming, and the city is Lisbon.


### 7.3 The user picks a smarter model

For a harder question the user switches to `gpt-5.4`.

In [11]:
ui["model"] = "gpt-5.4"
show(chat("We have a budget of 18,000 EUR. Hotel is 145 EUR per person per night for 3 nights, "
          "dinners are 60 EUR per person per night. How much is left for activities? "
          "Show the calculation in two lines."))

UI selection : model=gpt-5.4  reasoning_effort=medium
model_change : gpt-5.4-nano (medium) -> gpt-5.4 (medium)
model call   : model=gpt-5.4  reasoning_effort=medium  reasoning_tokens=148.0  output_tokens=218.0  ttft_ms=4446.0
reply (5.8s): Hotel: 145 x 12 x 3 = 5,220 EUR; dinners: 60 x 12 x 3 = 2,160 EUR; total = 7,380 EUR.  

18,000 EUR - 7,380 EUR = **10,620 EUR** left for activities.


### 7.4 Later, the user changes only the reasoning effort

The model stays on `gpt-5.4`. The user raises the reasoning effort to `high` for a scheduling
puzzle, then drops it to `low` for a simple follow-up. `session.model_change` reports the effort
change even though the model is the same.

In [12]:
ui["reasoning_effort"] = "high"
show(chat("Plan the 3 days: 4 workshops (A, B, C, D), at most 2 per day, A before C, B and D not on "
          "the same day, D not on day 1. Give one valid schedule as three short lines."))

UI selection : model=gpt-5.4  reasoning_effort=high
model_change : gpt-5.4 (medium) -> gpt-5.4 (high)
model call   : model=gpt-5.4  reasoning_effort=high  reasoning_tokens=128.0  output_tokens=153.0  ttft_ms=9076.0
reply (13.7s): Day 1: A, B  
Day 2: C  
Day 3: D


In [13]:
ui["reasoning_effort"] = "low"
show(chat("Remind me: what's my name and the team size? One line."))

UI selection : model=gpt-5.4  reasoning_effort=low
model_change : gpt-5.4 (high) -> gpt-5.4 (low)
model call   : model=gpt-5.4  reasoning_effort=low  reasoning_tokens=20.0  output_tokens=37.0  ttft_ms=7264.0
reply (9.0s): Priya, and the team size is 12.


### 7.5 Every turn ran on what the UI selected

Each row compares the UI selection sent with the request against the `assistant.usage` event the
agent emitted for that model call.

In [14]:
print(f"{'turn':<5}{'UI model':<15}{'UI effort':<11}{'used model':<15}{'used effort':<13}{'reasoning tok':>14}{'seconds':>9}")
for i, t in enumerate(TURNS, 1):
    for u in t["usage"]:
        print(f"{i:<5}{t['ui']['model']:<15}{t['ui']['reasoning_effort']:<11}{u.get('model'):<15}"
              f"{u.get('reasoningEffort'):<13}{int(u.get('reasoningTokens') or 0):>14}{t['seconds']:>9}")

mismatches = [(i, u) for i, t in enumerate(TURNS, 1) for u in t["usage"]
              if (u.get("model"), u.get("reasoningEffort")) != (t["ui"]["model"], t["ui"]["reasoning_effort"])]
print("\nEvery model call matched the UI selection." if not mismatches else f"\nMismatches: {mismatches}")

turn UI model       UI effort  used model     used effort   reasoning tok  seconds
1    gpt-5.4-mini   medium     gpt-5.4-mini   medium                  112     11.8
1    gpt-5.4-mini   medium     gpt-5.4-mini   medium                    0     11.8
2    gpt-5.4-nano   medium     gpt-5.4-nano   medium                    0      4.5
2    gpt-5.4-nano   medium     gpt-5.4-nano   medium                    0      4.5
3    gpt-5.4        medium     gpt-5.4        medium                  148      5.8
4    gpt-5.4        high       gpt-5.4        high                    128     13.7
5    gpt-5.4        low        gpt-5.4        low                      20      9.0

Every model call matched the UI selection.


### 7.6 No new version, no redeploy

The agent's versions, image digest and default model environment variable are exactly what they
were before the conversation. All switching happened inside the running session.

In [15]:
VERSIONS_AFTER = agent_versions()
print("Before:", VERSIONS_BEFORE)
print("After: ", VERSIONS_AFTER)
assert VERSIONS_AFTER == VERSIONS_BEFORE, "the agent's versions changed"
print("\nSame versions, same image, same environment - the switches were runtime-only.")

Before: [('4', 'active', 'github-copilot-runtime-model@sha256:afbac516f0d670b8990ce1f9c792e562c5cb680b409de776ca474d061d6e5470', 'gpt-5.4-mini')]
After:  [('4', 'active', 'github-copilot-runtime-model@sha256:afbac516f0d670b8990ce1f9c792e562c5cb680b409de776ca474d061d6e5470', 'gpt-5.4-mini')]

Same versions, same image, same environment - the switches were runtime-only.


### 7.7 The UI can only offer what the container allows

A model outside `AZURE_AI_ALLOWED_MODELS`, or an unknown reasoning effort, is rejected before it
reaches the session. The UI selection is restored afterwards so the session is unaffected.

In [16]:
saved = dict(ui)
ui.update({"model": "gpt-4o"})
print(chat("hello"))
ui.update({"model": saved["model"], "reasoning_effort": "extreme"})
print(chat("hello"))
ui.update(saved)

{'status': 400, 'error': {'error': 'invalid_model', 'message': "model must be one of ['gpt-5.4-nano', 'gpt-5.4-mini', 'gpt-5.4']"}}
{'status': 400, 'error': {'error': 'invalid_reasoning_effort', 'message': "reasoning_effort must be one of ['low', 'medium', 'high', 'xhigh']"}}


## Step 8 - Out-of-the-box telemetry

Telemetry from a hosted agent comes in three layers:

| Layer | What you get | Code needed |
|---|---|---|
| **Platform** | One `invoke_agent` request per invocation in Application Insights, from cloud role `agentsv2`, with agent name, version, session id and invocation id | None |
| **Container runtime** | Platform-injected environment variables, container stdout/stderr through the session log stream, and Python logging exported to the `traces` table (the platform injects `APPLICATIONINSIGHTS_CONNECTION_STRING`) | None |
| **Agent tracing** | `chat <model>` spans with model, reasoning effort, token usage and time to first token, plus `execute_tool <name>` spans, linked to the platform request | `tracing.py` in this container |

The Application Insights queries below can take a few minutes to return rows, because telemetry
ingestion is not instant.

### 8.1 Environment variables the platform injects

The platform injects these into every hosted agent container, with no configuration:

| Variable | Contents |
|---|---|
| `FOUNDRY_PROJECT_ENDPOINT` | Project endpoint |
| `FOUNDRY_PROJECT_ARM_ID` | Project ARM resource id |
| `FOUNDRY_AGENT_NAME` / `FOUNDRY_AGENT_VERSION` | Running agent name and version |
| `FOUNDRY_AGENT_SESSION_ID` | Session the container serves |
| `APPLICATIONINSIGHTS_CONNECTION_STRING` | Application Insights for the project, used by the tracing and log exporters |

The hosting library (`azure-ai-agentserver-core`) reads them at startup and logs a
`Platform environment` line, which the log stream in the next step shows.

### 8.2 Container logs: the session log stream

`GET {project}/agents/{agent}/versions/{version}/sessions/{session_id}:logstream` streams the
container's stdout and stderr for a session as server-sent events. The first event describes the
session (state, agent, last access); each following event has `stream` (`stdout`, `stderr`, or
`status`) and `message`. Connections close after 30 minutes, or after 2 minutes idle.

Below: the hosting library's startup line with the injected platform values, then the lines the
container logs when it creates the session and each time it switches the model. Python logging
goes to stderr, which is why these lines arrive on that stream.

In [17]:
def read_log_stream(seconds=25):
    url = (f"{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/versions/{AGENT_VERSION}"
           f"/sessions/{SESSION_ID}:logstream?api-version=v1")
    lines, started = [], time.time()
    try:
        with requests.get(url, headers=_headers(Accept="text/event-stream"), stream=True,
                          timeout=(10, seconds)) as resp:
            resp.raise_for_status()
            for line in resp.iter_lines(decode_unicode=True):
                if line and line.startswith("data:"):
                    lines.append(json.loads(line[5:].strip()))
                if time.time() - started > seconds:
                    break
    except requests.exceptions.ReadTimeout:
        pass  # the stream went idle; we have what was buffered
    return lines

LOG_LINES = read_log_stream()
header = LOG_LINES[0] if LOG_LINES else {}
print("Session:", {k: header.get(k) for k in ("session_state", "agent", "last_accessed")})
print(f"{len(LOG_LINES) - 1} log lines. Counts by stream:",
      {s: sum(1 for l in LOG_LINES if l.get("stream") == s) for s in ("stdout", "stderr", "status")})
print()
for l in LOG_LINES:
    message = l.get("message", "")
    if any(key in message for key in ("Platform environment", "Created session", "Switched model")):
        print(f"{l['timestamp'][:19]}  [{l['stream']}] {message.split(':', 2)[-1].strip()}")

Session: {'session_state': 'Running', 'agent': 'github-copilot-runtime-model', 'last_accessed': '2026-09-22T09:27:43.674+00:00'}
63 log lines. Counts by stream: {'stdout': 0, 'stderr': 60, 'status': 3}

2026-09-22T09:27:44  [stderr] Platform environment: is_hosted=True, agent_name=github-copilot-runtime-model, agent_version=4, port=8088, session_id={session-id}, sse_keepalive_interval=15, ws_ping_interval=30.0s
2026-09-22T09:27:47  [stderr] Created session: {session-id}
2026-09-22T09:28:12  [stderr] Switched model: {'model': 'gpt-5.4-mini', 'reasoning_effort': 'medium'} -> {'model': 'gpt-5.4-nano', 'reasoning_effort': 'medium'}
2026-09-22T09:28:29  [stderr] Switched model: {'model': 'gpt-5.4-nano', 'reasoning_effort': 'medium'} -> {'model': 'gpt-5.4', 'reasoning_effort': 'medium'}
2026-09-22T09:28:47  [stderr] Switched model: {'model': 'gpt-5.4', 'reasoning_effort': 'medium'} -> {'model': 'gpt-5.4', 'reasoning_effort': 'high'}
2026-09-22T09:29:06  [stderr] Switched model: {'model': 'gp

### 8.3 Application Insights: the platform's record of every invocation

The query helper below uses the Application Insights query API. The first query reads the
platform's own `invoke_agent` requests for this session. Nothing in the container produces these.

In [18]:
def kql(query, timespan="PT2H"):
    token = credential.get_token("https://api.applicationinsights.io/.default").token
    r = requests.post(f"https://api.applicationinsights.io/v1/apps/{APPINSIGHTS_APP_ID}/query",
                      headers={"Authorization": f"Bearer {token}"},
                      json={"query": query, "timespan": timespan})
    r.raise_for_status()
    table = r.json()["tables"][0]
    cols = [c["name"] for c in table["columns"]]
    return [dict(zip(cols, row)) for row in table["rows"]]

def kql_wait(query, min_rows, timeout=600, interval=30):
    # Poll until ingestion catches up.
    deadline = time.time() + timeout
    while True:
        rows = kql(query)
        if len(rows) >= min_rows or time.time() > deadline:
            return rows
        print(f"  {len(rows)} rows so far, waiting for ingestion...")
        time.sleep(interval)

platform_rows = kql_wait(f'''
requests
| where cloud_RoleName == "agentsv2"
| where tostring(customDimensions["azure.ai.agentserver.session_id"]) == "{SESSION_ID}"
| project timestamp, name, duration_ms = round(duration), resultCode, success,
          agent = tostring(customDimensions["gen_ai.agent.name"]),
          version = tostring(customDimensions["gen_ai.agent.version"])
| order by timestamp asc
''', min_rows=len(TURNS))

for row in platform_rows:
    print(f"{row['timestamp'][:19]}  {row['name']:<13} {row['agent']} v{row['version']}  "
          f"resultCode={row['resultCode']}  success={row['success']}  {row['duration_ms']} ms")

2026-09-22T09:27:44  invoke_agent  github-copilot-runtime-model v4  resultCode=0  success=True  8807 ms
2026-09-22T09:28:12  invoke_agent  github-copilot-runtime-model v4  resultCode=0  success=True  4030 ms
2026-09-22T09:28:29  invoke_agent  github-copilot-runtime-model v4  resultCode=0  success=True  5243 ms
2026-09-22T09:28:47  invoke_agent  github-copilot-runtime-model v4  resultCode=0  success=True  13209 ms
2026-09-22T09:29:05  invoke_agent  github-copilot-runtime-model v4  resultCode=0  success=True  8436 ms
2026-09-22T09:30:19  invoke_agent  github-copilot-runtime-model v4  resultCode=0  success=True  90 ms
2026-09-22T09:30:19  invoke_agent  github-copilot-runtime-model v4  resultCode=0  success=True  47 ms


The two invocations that took a few milliseconds are the rejected selections from 7.7. The
container answered HTTP 400, yet the platform recorded both as successful invocations. The platform
record shows that an invocation happened, how long it took and which session and version served
it; application-level errors show up only in the container's logs and spans.

### 8.4 Application Insights: each model call, with the model and reasoning effort used

These `chat <model>` spans come from this container's `tracing.py`. They share an `operation_Id`
with the platform request above, so the portal shows them in one trace. Here they show the runtime
switches from the telemetry side: the span name and attributes change with the UI selection.

In [19]:
chat_rows = kql_wait(f'''
dependencies
| where name startswith "chat "
| where tostring(customDimensions["gen_ai.conversation.id"]) == "{SESSION_ID}"
| project timestamp, name,
          reasoning_effort = tostring(customDimensions["gen_ai.request.reasoning_effort"]),
          input_tokens = toint(todouble(customDimensions["gen_ai.usage.input_tokens"])),
          output_tokens = toint(todouble(customDimensions["gen_ai.usage.output_tokens"])),
          reasoning_tokens = toint(todouble(customDimensions["gen_ai.usage.reasoning_tokens"])),
          ttft_ms = toint(todouble(customDimensions["gen_ai.server.time_to_first_token"]))
| order by timestamp asc
''', min_rows=sum(len(t["usage"]) for t in TURNS))

print(f"{'time':<21}{'span':<20}{'effort':<8}{'in':>8}{'out':>6}{'reasoning':>11}{'ttft ms':>9}")
for row in chat_rows:
    print(f"{row['timestamp'][:19]:<21}{row['name']:<20}{row['reasoning_effort']:<8}"
          f"{row['input_tokens']:>8}{row['output_tokens']:>6}{row['reasoning_tokens']:>11}{row['ttft_ms']:>9}")

time                 span                effort        in   out  reasoning  ttft ms
2026-09-22T09:27:52  chat gpt-5.4-mini   medium     11920   234        112     3455
2026-09-22T09:27:53  chat gpt-5.4-mini   medium     12212    20          0     1036
2026-09-22T09:28:14  chat gpt-5.4-nano   medium     12227   126          0     2129
2026-09-22T09:28:16  chat gpt-5.4-nano   medium     12448    15          0     1249
2026-09-22T09:28:34  chat gpt-5.4        medium     12595   218        148     4446
2026-09-22T09:29:00  chat gpt-5.4        high       12776   153        128     9076
2026-09-22T09:29:14  chat gpt-5.4        low        12873    37         20     7264


### 8.5 Application Insights: container logs as traces

Python logging from the container is exported to the `traces` table with the agent name as the
cloud role, so the "Switched model" lines from 8.2 are also queryable after the log stream is gone.

In [20]:
trace_rows = kql_wait(f'''
traces
| where cloud_RoleName == "{AGENT_NAME}"
| where timestamp >= datetime("{CONVERSATION_START}")
| where message startswith "Switched model"
| project timestamp, message
| order by timestamp asc
''', min_rows=1)

for row in trace_rows:
    print(f"{row['timestamp'][:19]}  {row['message']}")

2026-09-22T09:28:12  Switched model: {'model': 'gpt-5.4-mini', 'reasoning_effort': 'medium'} -> {'model': 'gpt-5.4-nano', 'reasoning_effort': 'medium'}
2026-09-22T09:28:29  Switched model: {'model': 'gpt-5.4-nano', 'reasoning_effort': 'medium'} -> {'model': 'gpt-5.4', 'reasoning_effort': 'medium'}
2026-09-22T09:28:47  Switched model: {'model': 'gpt-5.4', 'reasoning_effort': 'medium'} -> {'model': 'gpt-5.4', 'reasoning_effort': 'high'}
2026-09-22T09:29:06  Switched model: {'model': 'gpt-5.4', 'reasoning_effort': 'high'} -> {'model': 'gpt-5.4', 'reasoning_effort': 'low'}


### 8.6 Where to look in the portal

| Where | What it shows |
|---|---|
| Foundry portal, **Build > Agents > `github-copilot-runtime-model` > Traces** | The trace tree per invocation: platform `invoke_agent`, then the container's `chat <model>` and `execute_tool` spans |
| Foundry portal, agent **Monitor** tab (preview) | Token usage, latency, run success rate, and evaluation results over time |
| Application Insights, **Investigate > Transaction search / Performance** | The same requests and dependencies, plus CPU, memory, request rate and duration for right-sizing |
| Session log stream (8.2) | Live container stdout/stderr for one session |

To know when reading the raw data:

- The container's own spans report cloud role `unknown_service` because `tracing.py` does not set a
  service name. Set `OTEL_SERVICE_NAME` in the agent definition if you want a friendlier role name.

## Summary

**Runtime model and reasoning effort switching works without a new version or a redeploy**, with
these conditions:

- The platform does not provide the switch. Agent versions are immutable, so the container code must
  read the selection from the request body and apply it. With the Copilot SDK that is
  `session.set_model()`; other frameworks need their equivalent.
- The UI should send its selection on every request, because an idle session restarts its
  container and in-memory state is lost.
- Every selectable model must be deployed on the Foundry account, and the agent identity needs
  access to it. Restrict the choices with an allowlist.
- Supported reasoning effort values vary by model. `xhigh`, for example, is documented for
  `gpt-5.4` but not for `gpt-5.4-mini` or `gpt-5.4-nano`.

**Out-of-the-box telemetry** is the platform's per-invocation request record, the injected
environment variables, the session log stream and exported Python logs. Per-model-call detail
(model, reasoning effort, tokens, latency) needs a small tracing layer like `tracing.py`.

## Cleanup

Everything this lab created lives in its own resource group, so cleanup is one command. The cell
below is commented out; uncomment it when you want to tear the lab down.

In [ ]:
# sh(f'az group delete --name "{RESOURCE_GROUP}" --yes --no-wait')
# # The AI Services account is soft-deleted; purge it if you want the name back immediately:
# # sh(f'az cognitiveservices account purge --location "{LOCATION}" --resource-group "{RESOURCE_GROUP}" --name "{ACCOUNT_NAME}"')